# LeetCode 728 — Self Dividing Numbers bằng Python

**Link đề:** https://leetcode.com/problems/self-dividing-numbers/
**Độ khó:** Easy · **Chủ đề:** Math

```python
class Solution:
    def selfDividingNumbers(self, left: int, right: int) -> list[int]:
        ...
```

## Mục tiêu học
1. Hiểu định nghĩa **số tự chia hết** (self-dividing number) và cái bẫy **chữ số 0**.
2. Ôn toán tử `%` (chia lấy dư), `//` (chia lấy nguyên), `str()`, `int()`, `range(a, b + 1)`.
3. Tách bài thành 2 tầng: **hàm kiểm tra một số** + **vòng lặp lọc cả đoạn** — kỹ năng chia nhỏ bài toán.
4. Viết 3 cách: qua chuỗi → bóc chữ số bằng toán học → gọn bằng `all()` + list comprehension.
5. Trace, chứng minh, phân tích độ phức tạp, test đối chiếu và tự làm biến thể.

## 1. Đề bài

Một số là **self-dividing** nếu nó **chia hết cho mọi chữ số** mà nó chứa.

- Ví dụ `128`: `128 % 1 == 0`, `128 % 2 == 0`, `128 % 8 == 0` → self-dividing.
- Số chứa chữ số `0` **không** được coi là self-dividing (không chia cho 0 được).

Cho hai số nguyên `left`, `right`. Trả về **list tăng dần** mọi số self-dividing trong đoạn `[left, right]` (lấy cả hai đầu).

| left | right | Output |
|---:|---:|---|
| 1 | 22 | `[1,2,3,4,5,6,7,8,9,11,12,15,22]` |
| 47 | 85 | `[48,55,66,77]` |

**Constraints:** `1 <= left <= right <= 10^4`.

In [1]:
n = 128
for d in [1, 2, 8]:
    print(f"{n} % {d} = {n % d}")
print("=> 128 là self-dividing")
print()
n = 13
for d in [1, 3]:
    print(f"{n} % {d} = {n % d}")
print("=> 13 KHÔNG là self-dividing vì 13 % 3 != 0")

128 % 1 = 0
128 % 2 = 0
128 % 8 = 0
=> 128 là self-dividing

13 % 1 = 0
13 % 3 = 1
=> 13 KHÔNG là self-dividing vì 13 % 3 != 0


### Tính tay vài số trong đoạn `[10, 22]`

| Số | Chữ số | Kiểm tra | Self-dividing? |
|---:|---|---|:---:|
| 10 | 1, 0 | có chữ số 0 | ❌ |
| 11 | 1, 1 | `11 % 1 = 0` | ✅ |
| 12 | 1, 2 | `12 % 2 = 0` | ✅ |
| 13 | 1, 3 | `13 % 3 = 1` | ❌ |
| 15 | 1, 5 | `15 % 5 = 0` | ✅ |
| 20 | 2, 0 | có chữ số 0 | ❌ |
| 22 | 2, 2 | `22 % 2 = 0` | ✅ |

Khớp với output ví dụ 1 (`11, 12, 15, 22` xuất hiện; `10, 13, 20` không).

## 2. Nền tảng Python

### 2.1. `%` và `//` với số nguyên dương

- `a % b`: **phần dư** khi chia `a` cho `b`. `a % b == 0` ⟺ `a` chia hết cho `b`.
- `a // b`: **thương nguyên** (bỏ phần thập phân).
- Đặc biệt với `b = 10`: `x % 10` = chữ số **cuối**, `x // 10` = **bỏ** chữ số cuối.

In [2]:
x = 128
print(x % 10)     # 8   chữ số cuối
print(x // 10)    # 12  bỏ chữ số cuối
print(x / 10)     # 12.8  (phép / cho số thực — KHÔNG dùng để bóc chữ số)
print(22 % 2 == 0, 13 % 3 == 0)

8
12
12.8
True False


### 2.2. Chia cho 0 gây lỗi

`x % 0` ném `ZeroDivisionError`. Đó là lý do phải kiểm tra chữ số 0 **trước** khi dùng `%`. Cell dưới bắt lỗi bằng `try/except` để xem thông báo mà notebook vẫn chạy tiếp.

In [3]:
try:
    print(10 % 0)
except ZeroDivisionError as e:
    print("Lỗi:", type(e).__name__, "-", e)

Lỗi: ZeroDivisionError - integer modulo by zero


### 2.3. Đổi số thành chuỗi và ngược lại

`str(128)` → `"128"`; duyệt chuỗi bằng `for ch in "128"` cho từng **ký tự** `'1'`, `'2'`, `'8'`. Muốn tính toán phải đổi lại số: `int('8')` → `8`.

In [4]:
s = str(128)
for ch in s:
    print(repr(ch), "->", int(ch), type(int(ch)).__name__)
print("'0' in '105' ?", '0' in '105')

'1' -> 1 int
'2' -> 2 int
'8' -> 8 int
'0' in '105' ? True


### 2.4. `range(left, right + 1)` — đoạn có lấy cả hai đầu

`range(a, b)` dừng **trước** `b`. Đề nói "inclusive" nên phải viết `range(left, right + 1)`.

In [5]:
print(list(range(1, 5)))       # [1, 2, 3, 4]      -> thiếu 5
print(list(range(1, 5 + 1)))   # [1, 2, 3, 4, 5]

[1, 2, 3, 4]
[1, 2, 3, 4, 5]


### 2.5. `all()` và list comprehension

- `all(iterable)` trả `True` nếu **mọi** phần tử đều đúng (và `True` với iterable rỗng). Nó **dừng sớm** ngay khi gặp phần tử sai.
- List comprehension `[f(x) for x in xs if cond(x)]` tạo list mới gọn trong một dòng.

In [6]:
print(all([True, True, True]))     # True
print(all([True, False, True]))    # False
print(all([]))                     # True (chú ý!)
squares_of_even = [x * x for x in range(6) if x % 2 == 0]
print(squares_of_even)             # [0, 4, 16]

True
False
True
[0, 4, 16]


## 3. Chia bài toán thành 2 tầng

Đây là kỹ năng quan trọng hơn cả thuật toán:

1. **Tầng nhỏ:** `is_self_dividing(n) -> bool` — chỉ lo **một** số.
2. **Tầng lớn:** duyệt `n` từ `left` đến `right`, giữ lại số nào `is_self_dividing(n)` đúng.

Tầng lớn luôn giống nhau; ta sẽ thử nhiều cách viết tầng nhỏ. Vì duyệt `n` tăng dần nên list kết quả **tự động tăng dần** — không cần `sort`.

Bài này không có "Brute Force chậm vs tối ưu nhanh" rõ rệt: constraints nhỏ (`≤ 10^4` số, mỗi số ≤ 5 chữ số) nên duyệt hết là cách tự nhiên và đủ nhanh. Trọng tâm là **kiểm tra đúng** từng số.

## 4. Cách 1 — Kiểm tra qua chuỗi

Ý tưởng dịch thẳng từ đề:
- Đổi `n` thành chuỗi.
- Với từng ký tự `ch`: nếu là `'0'` → sai ngay; nếu `n % int(ch) != 0` → sai ngay.
- Qua hết → đúng.

In [7]:
def is_self_dividing_str(n: int) -> bool:
    for ch in str(n):
        d = int(ch)
        if d == 0:
            return False
        if n % d != 0:
            return False
    return True

for n in [128, 13, 10, 22, 7]:
    print(n, "->", is_self_dividing_str(n))

128 -> True
13 -> False
10 -> False
22 -> True
7 -> True


In [8]:
def self_dividing_str(left: int, right: int) -> list[int]:
    result = []
    for n in range(left, right + 1):
        if is_self_dividing_str(n):
            result.append(n)
    return result

print(self_dividing_str(1, 22))
print(self_dividing_str(47, 85))

[1, 2, 3, 4, 5, 6, 7, 8, 9, 11, 12, 15, 22]
[48, 55, 66, 77]


Cách 1 dễ đọc. Nhược điểm nhỏ: mỗi số tạo một chuỗi mới và đổi ký tự ↔ số nhiều lần. Cách 2 làm việc trực tiếp trên số.

## 5. Cách 2 — Bóc chữ số bằng `% 10` và `// 10` (lời giải chính)

Giữ bản sao `x = n` để bóc dần, còn `n` giữ nguyên để làm số bị chia:

```text
x = 128 -> d = 8, 128 % 8 = 0 ✔, x = 12
x = 12  -> d = 2, 128 % 2 = 0 ✔, x = 1
x = 1   -> d = 1, 128 % 1 = 0 ✔, x = 0  -> dừng, True
```

Điểm dễ sai: phải lấy `n % d` (số **gốc**), **không phải** `x % d` (số đã bị cắt).

In [9]:
def is_self_dividing_math(n: int) -> bool:
    x = n
    while x > 0:
        d = x % 10
        if d == 0 or n % d != 0:
            return False
        x //= 10
    return True

for n in [128, 13, 10, 22, 7, 1000, 9999]:
    print(n, "->", is_self_dividing_math(n))

128 -> True
13 -> False
10 -> False
22 -> True
7 -> True
1000 -> False
9999 -> True


### Vì sao `d == 0 or n % d != 0` không bị lỗi chia 0?

Toán tử `or` **đánh giá ngắn mạch** (short-circuit): nếu vế trái `d == 0` đã `True`, Python **không** tính vế phải. Nên `n % 0` không bao giờ được thực thi. Đổi thứ tự hai vế sẽ gây lỗi!

In [10]:
d, n = 0, 10
print(d == 0 or n % d != 0)       # True, an toàn

try:
    print(n % d != 0 or d == 0)   # vế trái tính trước -> lỗi
except ZeroDivisionError:
    print("Đổi thứ tự -> ZeroDivisionError!")

True
Đổi thứ tự -> ZeroDivisionError!


In [11]:
def self_dividing_math(left: int, right: int) -> list[int]:
    result = []
    for n in range(left, right + 1):
        if is_self_dividing_math(n):
            result.append(n)
    return result

print(self_dividing_math(1, 22))
print(self_dividing_math(47, 85))

[1, 2, 3, 4, 5, 6, 7, 8, 9, 11, 12, 15, 22]
[48, 55, 66, 77]


## 6. Trace từng bước

Bảng trace cho `n = 128` và `n = 48` và `n = 102`:

| n | x (trước) | d = x % 10 | d == 0? | n % d | Quyết định | x (sau) |
|---:|---:|---:|:---:|---:|---|---:|
| 128 | 128 | 8 | không | 0 | tiếp | 12 |
| 128 | 12 | 2 | không | 0 | tiếp | 1 |
| 128 | 1 | 1 | không | 0 | tiếp | 0 → True |
| 48 | 48 | 8 | không | 0 | tiếp | 4 |
| 48 | 4 | 4 | không | 0 | tiếp | 0 → True |
| 102 | 102 | 2 | không | 0 | tiếp | 10 |
| 102 | 10 | 0 | **có** | — | **False** | — |

Chú ý `102`: chữ số `0` nằm giữa nhưng vẫn bị phát hiện vì ta bóc **mọi** chữ số.

In [12]:
def trace_self_dividing(n: int) -> bool:
    x = n
    step = 1
    print(f"Kiểm tra n = {n}")
    while x > 0:
        d = x % 10
        if d == 0:
            print(f"  bước {step}: x={x}, d=0 -> có chữ số 0 => False")
            return False
        r = n % d
        print(f"  bước {step}: x={x}, d={d}, n % d = {r}", "-> tiếp" if r == 0 else "-> không chia hết => False")
        if r != 0:
            return False
        x //= 10
        step += 1
    print("  hết chữ số => True")
    return True

for n in [128, 48, 102, 49]:
    trace_self_dividing(n)

Kiểm tra n = 128
  bước 1: x=128, d=8, n % d = 0 -> tiếp
  bước 2: x=12, d=2, n % d = 0 -> tiếp
  bước 3: x=1, d=1, n % d = 0 -> tiếp
  hết chữ số => True
Kiểm tra n = 48
  bước 1: x=48, d=8, n % d = 0 -> tiếp
  bước 2: x=4, d=4, n % d = 0 -> tiếp
  hết chữ số => True
Kiểm tra n = 102
  bước 1: x=102, d=2, n % d = 0 -> tiếp
  bước 2: x=10, d=0 -> có chữ số 0 => False
Kiểm tra n = 49
  bước 1: x=49, d=9, n % d = 4 -> không chia hết => False


### Trace tầng lớn: đoạn `[47, 55]`

Xem số nào được giữ, số nào bị loại và **lý do**.

In [13]:
def trace_range(left: int, right: int) -> list[int]:
    result = []
    for n in range(left, right + 1):
        reason = None
        x = n
        while x > 0:
            d = x % 10
            if d == 0:
                reason = "chứa 0"; break
            if n % d != 0:
                reason = f"{n} % {d} = {n % d}"; break
            x //= 10
        if reason is None:
            result.append(n)
            print(f"{n}: GIỮ   -> result = {result}")
        else:
            print(f"{n}: loại ({reason})")
    return result

trace_range(47, 55)

47: loại (47 % 7 = 5)
48: GIỮ   -> result = [48]
49: loại (49 % 9 = 4)
50: loại (chứa 0)
51: loại (51 % 5 = 1)
52: loại (52 % 5 = 2)
53: loại (53 % 3 = 2)
54: loại (54 % 4 = 2)
55: GIỮ   -> result = [48, 55]


[48, 55]

## 7. Cách 3 — Viết gọn bằng `all()` + list comprehension

Ghép điều kiện "mọi chữ số `d` đều khác 0 và chia hết" thành một biểu thức. `all()` dừng sớm ở chữ số đầu tiên vi phạm, nên vẫn hiệu quả.

In [14]:
def self_dividing_pythonic(left: int, right: int) -> list[int]:
    return [n for n in range(left, right + 1)
            if all(ch != '0' and n % int(ch) == 0 for ch in str(n))]

print(self_dividing_pythonic(1, 22))
print(self_dividing_pythonic(47, 85))

[1, 2, 3, 4, 5, 6, 7, 8, 9, 11, 12, 15, 22]
[48, 55, 66, 77]


Biểu thức `ch != '0' and n % int(ch) == 0` cũng dựa vào **ngắn mạch** của `and`: nếu `ch == '0'`, vế phải không được tính → không chia cho 0.

Ngoài ra `filter(hàm, iterable)` là cách viết khác của tầng lớn:

In [15]:
print(list(filter(is_self_dividing_math, range(1, 23))))

[1, 2, 3, 4, 5, 6, 7, 8, 9, 11, 12, 15, 22]


## 8. Code nộp LeetCode

### Bản chính (bóc chữ số bằng toán)

In [16]:
class Solution:
    def selfDividingNumbers(self, left: int, right: int) -> list[int]:
        def ok(n: int) -> bool:
            x = n
            while x > 0:
                d = x % 10
                if d == 0 or n % d != 0:
                    return False
                x //= 10
            return True

        return [n for n in range(left, right + 1) if ok(n)]


sol = Solution()
print(sol.selfDividingNumbers(1, 22))
print(sol.selfDividingNumbers(47, 85))

[1, 2, 3, 4, 5, 6, 7, 8, 9, 11, 12, 15, 22]
[48, 55, 66, 77]


### Bản chuỗi và bản một dòng

Đều được chấp nhận. Trên LeetCode đổi tên class thành `Solution`.

In [17]:
class SolutionString:
    def selfDividingNumbers(self, left: int, right: int) -> list[int]:
        result = []
        for n in range(left, right + 1):
            s = str(n)
            if '0' in s:
                continue
            if all(n % int(ch) == 0 for ch in s):
                result.append(n)
        return result


class SolutionOneLine:
    def selfDividingNumbers(self, left: int, right: int) -> list[int]:
        return [n for n in range(left, right + 1)
                if all(c != '0' and n % int(c) == 0 for c in str(n))]


print(SolutionString().selfDividingNumbers(1, 22) == SolutionOneLine().selfDividingNumbers(1, 22))

True


Ở `SolutionString`, câu `continue` nghĩa là "bỏ qua phần còn lại của vòng hiện tại, sang `n` kế tiếp". Ta lọc chữ số 0 bằng `'0' in s` trước, nên trong `all(...)` không còn nguy cơ chia 0.

## 9. Vì sao đúng?

**Hàm kiểm tra (Cách 2).** Invariant của vòng `while`: *mọi chữ số đã bị bóc ra khỏi `x` đều khác 0 và chia hết `n`.*
- Ban đầu chưa bóc chữ số nào → đúng hiển nhiên.
- Mỗi vòng bóc chữ số cuối `d` của `x`; nếu `d` vi phạm ta trả `False` ngay (đúng, vì `n` có một chữ số xấu). Nếu không, invariant được giữ.
- Vì `x % 10` rồi `x //= 10` lần lượt lấy **mọi** chữ số của `n` (từ phải sang trái), khi `x == 0` thì mọi chữ số đã được kiểm → trả `True` đúng.

**Tầng lớn.** Duyệt đúng mọi `n` trong `[left, right]` (nhờ `right + 1`), thêm `n` khi và chỉ khi hàm kiểm tra đúng; thứ tự duyệt tăng nên list tăng dần.

## 10. Độ phức tạp

Gọi `D = right - left + 1` (số lượng số cần xét), `k` = số chữ số của `right` (`k = ⌊log10(right)⌋ + 1 ≤ 5`).

| Cách | Time | Space phụ (không tính output) |
|---|---|---|
| 1. Chuỗi | `O(D · k)` | `O(k)` cho chuỗi tạm |
| 2. `% 10` / `// 10` | `O(D · k)` | `O(1)` |
| 3. `all()` + comprehension | `O(D · k)` | `O(k)` cho chuỗi tạm |

Với constraints: tối đa `10^4 · 5 = 5·10^4` phép kiểm — rất nhanh. Output có thể dài tới `D` phần tử (`O(D)`), nhưng đó là bộ nhớ bắt buộc của kết quả.

Số chữ số `k` tỉ lệ `log10(n)`, nên có thể viết gọn là `O(D log right)`.

## 11. Bộ test

In [18]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS  {name}")

sol = Solution()
check("đề 1", sol.selfDividingNumbers(1, 22), [1, 2, 3, 4, 5, 6, 7, 8, 9, 11, 12, 15, 22])
check("đề 2", sol.selfDividingNumbers(47, 85), [48, 55, 66, 77])
check("một số, có", sol.selfDividingNumbers(128, 128), [128])
check("một số, không", sol.selfDividingNumbers(13, 13), [])
check("toàn chứa 0", sol.selfDividingNumbers(100, 110), [])
check("biên dưới", sol.selfDividingNumbers(1, 1), [1])
check("biên trên", sol.selfDividingNumbers(9990, 10000), [9999])
check("0 ở giữa", sol.selfDividingNumbers(101, 102), [])

PASS  đề 1
PASS  đề 2
PASS  một số, có
PASS  một số, không
PASS  toàn chứa 0
PASS  biên dưới
PASS  biên trên
PASS  0 ở giữa


Vì sao "biên trên" là `[9999]`? Tính tay:
- `9990` và `10000` chứa chữ số 0 → loại.
- `9991..9998` có dạng `999k`; muốn chia hết cho chữ số 9 thì tổng chữ số `27 + k` phải chia hết cho 9 → `k = 0` hoặc `9`. Chỉ còn `9999 = 9 · 1111` ✔.

Mẹo viết test: tự **tính tay** expected và ghi thẳng giá trị rõ ràng; nếu không chắc, đối chiếu thêm với một cài đặt khác (cell dưới).

In [19]:
expected_top = [n for n in range(9990, 10001) if is_self_dividing_str(n)]
print("expected tính bằng cách 1:", expected_top)
check("biên trên (đối chiếu cách 1)", sol.selfDividingNumbers(9990, 10000), expected_top)

expected tính bằng cách 1: [9999]
PASS  biên trên (đối chiếu cách 1)


In [20]:
import random

rng = random.Random(728)
impls = [self_dividing_str, self_dividing_math, self_dividing_pythonic,
         Solution().selfDividingNumbers, SolutionString().selfDividingNumbers, SolutionOneLine().selfDividingNumbers]
for _ in range(300):
    a = rng.randint(1, 10_000)
    c = rng.randint(a, min(10_000, a + 300))
    outs = [f(a, c) for f in impls]
    assert all(o == outs[0] for o in outs), (a, c)
full = Solution().selfDividingNumbers(1, 10_000)
print("300 đoạn ngẫu nhiên: 6 cài đặt khớp nhau.")
print("Số lượng self-dividing trong [1, 10^4]:", len(full), "| 10 số cuối:", full[-10:])

300 đoạn ngẫu nhiên: 6 cài đặt khớp nhau.
Số lượng self-dividing trong [1, 10^4]: 339 | 10 số cuối: [9324, 9333, 9396, 9432, 9612, 9648, 9666, 9864, 9936, 9999]


## 12. Lỗi người mới hay gặp

### Lỗi 1: Quên kiểm tra chữ số 0 → `ZeroDivisionError`

In [21]:
def wrong_no_zero_check(n):
    x = n
    while x > 0:
        d = x % 10
        if n % d != 0:           # d có thể bằng 0!
            return False
        x //= 10
    return True

try:
    wrong_no_zero_check(10)
except ZeroDivisionError:
    print("wrong_no_zero_check(10) -> ZeroDivisionError")
print("đúng:", is_self_dividing_math(10))

wrong_no_zero_check(10) -> ZeroDivisionError
đúng: False


**Sửa:** `if d == 0 or n % d != 0` (vế `d == 0` đứng trước).

### Lỗi 2: Chia cho `x` (số đang bị cắt) thay vì `n` (số gốc)

In [22]:
def wrong_divide_x(n):
    x = n
    while x > 0:
        d = x % 10
        if d == 0 or x % d != 0:   # SAI: x đã bị cắt
            return False
        x //= 10
    return True

print("21 sai :", wrong_divide_x(21), "| đúng:", is_self_dividing_math(21))

21 sai : True | đúng: False


Với `21`: bản sai ở bước 2 có `x = 2`, kiểm `2 % 2 = 0` ✔ nên trả True; nhưng thật ra `21 % 2 = 1` → phải False. Còn bao nhiêu số bị sai như vậy? Thay vì đoán, **cho máy dò** bằng cách so với cài đặt đúng:

In [23]:
bad = [n for n in range(1, 200) if wrong_divide_x(n) != is_self_dividing_math(n)]
print("Các số mà wrong_divide_x cho sai:", bad[:15])
n = bad[0]
print(f"Ví dụ n = {n}: sai -> {wrong_divide_x(n)}, đúng -> {is_self_dividing_math(n)}")

Các số mà wrong_divide_x cho sai: [21, 25, 31, 32, 35, 41, 42, 45, 51, 52, 61, 62, 63, 64, 65]
Ví dụ n = 21: sai -> True, đúng -> False


Lỗi chỉ lộ ra khi một chữ số phía trước **chia hết phần đã bị cắt** nhưng **không chia hết số gốc**. **Sửa:** luôn dùng `n % d`.

Bài học phụ: phản ví dụ hãy để máy tìm bằng cách so với một cài đặt chắc chắn đúng.

### Lỗi 3: `range(left, right)` — mất số `right`

In [24]:
def wrong_range(left, right):
    return [n for n in range(left, right) if is_self_dividing_math(n)]

print("sai :", wrong_range(1, 22)[-3:], "| đúng:", self_dividing_math(1, 22)[-3:])

sai : [11, 12, 15] | đúng: [12, 15, 22]


### Lỗi 4: So sánh ký tự với số: `if ch == 0`

`ch` là chuỗi `'0'`, còn `0` là số → `'0' == 0` luôn `False`, nên chữ số 0 lọt qua và lại gây chia cho 0.

In [25]:
print("'0' == 0 ->", '0' == 0)
print("'0' == '0' ->", '0' == '0')
print("int('0') == 0 ->", int('0') == 0)

'0' == 0 -> False
'0' == '0' -> True
int('0') == 0 -> True


### Lỗi 5: Dùng `x / 10` thay vì `x // 10`

`/` cho số thực, vòng `while x > 0` sẽ chạy hàng trăm lần với các giá trị như `12.8, 1.28, 0.128, ...` và `x % 10` ra số thực. Luôn dùng `//` khi bóc chữ số.

## 13. Mẹo nhận dạng pattern

- Đề nói về **"chữ số của số"** → nghĩ tới cặp `x % 10` / `x // 10` (hoặc `str(n)` nếu không cấm).
- Đề "liệt kê mọi số trong đoạn thỏa điều kiện" + đoạn nhỏ (`≤ 10^5`–`10^6`) → **duyệt hết + hàm kiểm tra**. Không cần thuật toán phức tạp.
- Thấy phép chia/modulo cho một giá trị **có thể bằng 0** → kiểm tra 0 trước, tận dụng ngắn mạch `or`/`and`.
- Các bài cùng họ: 9 Palindrome Number, 258 Add Digits, 202 Happy Number, 1281 Subtract Product and Sum of Digits, 2520 Count the Digits That Divide a Number.

## 14. Biến thể

### 14.1. LeetCode 2520 — Count the Digits That Divide a Number

Cho `num` (không chứa chữ số 0), đếm số chữ số `d` của `num` mà `num % d == 0` (đếm cả lặp lại). Cùng kỹ thuật bóc chữ số.

In [26]:
def count_digits(num: int) -> int:
    count, x = 0, num
    while x > 0:
        d = x % 10
        if num % d == 0:
            count += 1
        x //= 10
    return count

print(count_digits(7), count_digits(121), count_digits(1248))   # 1 2 4

1 2 4


### 14.2. Đếm thay vì liệt kê, và tối ưu bằng mảng đánh dấu khi có nhiều truy vấn

Nếu phải trả lời **nhiều** câu hỏi "có bao nhiêu số self-dividing trong `[l, r]`", ta tính trước một lần cho `1..10^4` rồi dùng **prefix sum** (cùng kỹ thuật bài 724): mỗi truy vấn `O(1)`.

In [27]:
LIMIT = 10_000
prefix_cnt = [0] * (LIMIT + 1)
for n in range(1, LIMIT + 1):
    prefix_cnt[n] = prefix_cnt[n - 1] + (1 if is_self_dividing_math(n) else 0)

def count_in_range(l: int, r: int) -> int:
    return prefix_cnt[r] - prefix_cnt[l - 1]

print(count_in_range(1, 22), len(self_dividing_math(1, 22)))
print(count_in_range(47, 85), len(self_dividing_math(47, 85)))

13 13
4 4


### 14.3. Mỗi chữ số chỉ cần kiểm một lần

Số `1111` có bốn chữ số `1` nhưng chỉ cần kiểm `1` một lần. Dùng `set(str(n))` để lấy các chữ số **khác nhau**. Không thay đổi Big-O nhưng là thói quen tốt khi kiểm tra đắt.

In [28]:
def is_self_dividing_set(n: int) -> bool:
    digits = set(str(n))
    if '0' in digits:
        return False
    return all(n % int(c) == 0 for c in digits)

print(set(str(1248)), set(str(1111)))
print(all(is_self_dividing_set(n) == is_self_dividing_math(n) for n in range(1, 10_001)))

{'1', '8', '2', '4'} {'1'}


True


## 15. Bài tập

### Bài 1 — Đoán output (không chạy)
`is_self_dividing_math(n)` trả gì với `n = 24`, `36`, `105`, `333`? Ghi ra giấy rồi xem đáp án.

### Bài 2 — Đếm số chữ số
Viết `digit_count(n)` trả số chữ số của `n >= 1` **không dùng `str`**.

In [29]:
def digit_count(n: int) -> int:
    # TODO: dùng // 10 trong vòng while
    pass

# assert digit_count(7) == 1
# assert digit_count(128) == 3
# assert digit_count(10000) == 5

### Bài 3 — Điền chỗ trống hàm kiểm tra

In [30]:
def exercise_is_self_dividing(n: int) -> bool:
    x = n
    while x > 0:
        # TODO 1: lấy chữ số cuối d
        # TODO 2: nếu d == 0 hoặc n không chia hết cho d -> False (chú ý thứ tự!)
        # TODO 3: bỏ chữ số cuối khỏi x
        pass
    return True

# assert exercise_is_self_dividing(128) == True
# assert exercise_is_self_dividing(102) == False
# assert exercise_is_self_dividing(21) == False

### Bài 4 — Số "chia hết cho tổng chữ số" (Harshad / Niven number)
Liệt kê mọi `n` trong `[left, right]` chia hết cho **tổng** các chữ số của nó.

In [31]:
def harshad_numbers(left: int, right: int) -> list[int]:
    # TODO
    pass

# assert harshad_numbers(10, 30) == [10, 12, 18, 20, 21, 24, 27, 30]

### Bài 5 — Self-dividing nghiêm ngặt
Một số gọi là *strict self-dividing* nếu là self-dividing **và** các chữ số **đôi một khác nhau**. Liệt kê trong `[left, right]`.

In [32]:
def strict_self_dividing(left: int, right: int) -> list[int]:
    # TODO
    pass

# assert strict_self_dividing(1, 40) == [1, 2, 3, 4, 5, 6, 7, 8, 9, 12, 15, 24, 36]

### Bài 6 — Mini challenge: số self-dividing tiếp theo
Viết `next_self_dividing(n)` trả số self-dividing **nhỏ nhất lớn hơn `n`**. Sau đó dùng nó để tìm số self-dividing đầu tiên lớn hơn `10^4`.

In [33]:
def next_self_dividing(n: int) -> int:
    # TODO
    pass

# assert next_self_dividing(9) == 11
# assert next_self_dividing(22) == 24

## 16. Gợi ý 3 mức

| Bài | Mức 1 — hướng nghĩ | Mức 2 — cấu trúc | Mức 3 — gần code |
|---|---|---|---|
| 1 | Bóc từng chữ số | Có 0? Chia hết từng chữ số? | `24%4, 24%2`; `36%6, 36%3`; `105` có 0; `333%3` |
| 2 | Mỗi lần `// 10` mất 1 chữ số | Đếm số vòng tới khi về 0 | `while n > 0: n //= 10; cnt += 1` |
| 3 | Đúng như Cách 2 | `d = x % 10` | `if d == 0 or n % d != 0: return False`; `x //= 10` |
| 4 | Thay "từng chữ số" bằng "tổng chữ số" | Hàm `digit_sum(n)` | `[n for n in range(l, r+1) if n % digit_sum(n) == 0]` |
| 5 | Hai điều kiện cùng lúc | So `len(set(s))` với `len(s)` | `ok(n) and len(set(str(n))) == len(str(n))` |
| 6 | Thử lần lượt `n+1, n+2, ...` | Vòng `while` tới khi gặp | `m = n + 1; while not ok(m): m += 1` |

## 17. Đáp án tham khảo

**Bài 1:** `24` → True (`24%4 = 0`, `24%2 = 0`); `36` → True (`36%6 = 0`, `36%3 = 0`); `105` → False (chứa 0); `333` → True (`333%3 = 0`). Cell dưới kiểm chứng toàn bộ.

In [34]:
print({n: is_self_dividing_math(n) for n in [24, 36, 105, 333]})


def answer_digit_count(n):
    cnt = 0
    while n > 0:
        n //= 10
        cnt += 1
    return cnt


def answer_is_self_dividing(n):
    x = n
    while x > 0:
        d = x % 10
        if d == 0 or n % d != 0:
            return False
        x //= 10
    return True


def digit_sum(n):
    s = 0
    while n > 0:
        s += n % 10
        n //= 10
    return s


def answer_harshad(left, right):
    return [n for n in range(left, right + 1) if n % digit_sum(n) == 0]


def answer_strict(left, right):
    return [n for n in range(left, right + 1)
            if answer_is_self_dividing(n) and len(set(str(n))) == len(str(n))]


def answer_next_self_dividing(n):
    m = n + 1
    while not answer_is_self_dividing(m):
        m += 1
    return m


assert answer_digit_count(7) == 1 and answer_digit_count(128) == 3 and answer_digit_count(10000) == 5
assert answer_is_self_dividing(128) and not answer_is_self_dividing(102) and not answer_is_self_dividing(21)
assert answer_harshad(10, 30) == [10, 12, 18, 20, 21, 24, 27, 30]
assert answer_strict(1, 40) == [1, 2, 3, 4, 5, 6, 7, 8, 9, 12, 15, 24, 36]
assert answer_next_self_dividing(9) == 11
assert answer_next_self_dividing(22) == 24
print("Số self-dividing đầu tiên > 10^4:", answer_next_self_dividing(10_000))
print("Tất cả đáp án PASS")

{24: True, 36: True, 105: False, 333: True}
Số self-dividing đầu tiên > 10^4: 11111
Tất cả đáp án PASS


Nhận xét bài 4: số Harshad **có thể** chứa chữ số 0 (ví dụ `10`, `20`, `30`) vì ta chia cho **tổng** chữ số, tổng này luôn ≥ 1 với `n ≥ 1`. So sánh với bài gốc để thấy điều kiện "chữ số 0" phụ thuộc vào việc ta chia cho cái gì.

Nhận xét bài 5: `11`, `22`, `33` bị loại vì chữ số lặp; `48` (`48%4 = 0`, `48%8 = 0`) không nằm trong `[1, 40]`.

## 18. Template nhớ nhanh + Checklist submit

```python
class Solution:
    def selfDividingNumbers(self, left: int, right: int) -> list[int]:
        def ok(n):
            x = n
            while x > 0:
                d = x % 10                       # chữ số cuối
                if d == 0 or n % d != 0:         # 0 kiểm TRƯỚC, chia số GỐC n
                    return False
                x //= 10                         # bỏ chữ số cuối
            return True
        return [n for n in range(left, right + 1) if ok(n)]
```

**Câu thần chú:** *"tách hàm kiểm tra — % 10 lấy, // 10 bỏ — 0 trước, chia số gốc — range tới right + 1."*

**Checklist trước khi submit:**
- [ ] `range(left, right + 1)` (đoạn lấy cả hai đầu)?
- [ ] Kiểm `d == 0` **trước** phép `%` (ngắn mạch `or`)?
- [ ] Chia **số gốc** `n`, không phải bản sao `x` đang bị cắt?
- [ ] Dùng `//` chứ không phải `/`?
- [ ] Nếu dùng chuỗi: so sánh `ch == '0'` (ký tự), không phải `ch == 0`?
- [ ] Đã test đoạn chỉ một số, đoạn toàn số chứa 0 (kết quả `[]`)?